In [0]:
# ============================================================
# CITYFIX - WEEK 7 GOLD LAYER
# Gold tables built from silver_trusted_requests
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Source and target
SOURCE_TABLE = "cityfix.cityfix.silver_trusted_requests"
GOLD_SCHEMA = "cityfix.gold"

print("Source:", SOURCE_TABLE)
print("Target schema:", GOLD_SCHEMA)

# Create Gold schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

# Read Trusted Silver only
silver = spark.table(SOURCE_TABLE)

print("Trusted Silver rows:", silver.count())
silver.printSchema()

Source: cityfix.cityfix.silver_trusted_requests
Target schema: cityfix.gold
Trusted Silver rows: 179701
root
 |-- physical_record_key: string (nullable = true)
 |-- unique_key: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- due_date: string (nullable = true)
 |-- closed_date: string (nullable = true)
 |-- agency_code: string (nullable = true)
 |-- agency_name_raw: string (nullable = true)
 |-- complaint_category: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- descriptor: string (nullable = true)
 |-- borough_code: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- resolution_description: string (nullable = true)
 |-- sla_hours: string 

In [0]:
# ============================================================
# 1. DIM_DATE
# ============================================================

date_df = (
    silver
    .filter(F.col("created_ts").isNotNull())
    .select(F.to_date("created_ts").alias("full_date"))
    .distinct()
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("week_of_year", F.weekofyear("full_date"))
    .withColumn("day_of_month", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .orderBy("full_date")
)

date_df.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.dim_date"
)

print("dim_date created")

dim_date created


In [0]:
# ============================================================
# 2. DIM_TIME_BAND
# ============================================================

time_band_df = spark.createDataFrame(
    [
        (1, "Night", "00:00-05:59", 0, 5),
        (2, "Morning", "06:00-11:59", 6, 11),
        (3, "Afternoon", "12:00-17:59", 12, 17),
        (4, "Evening", "18:00-23:59", 18, 23)
    ],
    [
        "time_band_key",
        "time_band",
        "hour_range",
        "start_hour",
        "end_hour"
    ]
)

time_band_df.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.dim_time_band"
)

print("dim_time_band created")

dim_time_band created


In [0]:
# ============================================================
# 3. DIM_AGENCY
# ============================================================

agency_df = (
    silver
    .select(
        F.col("agency_code_std").alias("agency_code"),
        F.col("agency_name_std").alias("agency_name")
    )
    .filter(F.col("agency_code").isNotNull())
    .dropDuplicates(["agency_code"])
)

agency_window = Window.orderBy("agency_code")

agency_df = (
    agency_df
    .withColumn("agency_key", F.row_number().over(agency_window))
    .select(
        "agency_key",
        "agency_code",
        "agency_name"
    )
)

agency_df.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.dim_agency"
)

print("dim_agency created")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_agency created


In [0]:
# ============================================================
# 4. DIM_COMPLAINT_CATEGORY
# ============================================================

category_df = (
    silver
    .select(
        F.col("complaint_category_std").alias("complaint_category"),
        F.col("complaint_type_std").alias("complaint_type"),
        F.col("descriptor_std").alias("descriptor")
    )
    .dropDuplicates()
)

category_window = Window.orderBy(
    "complaint_category",
    "complaint_type",
    "descriptor"
)

category_df = (
    category_df
    .withColumn(
        "complaint_category_key",
        F.row_number().over(category_window)
    )
)

category_df.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.dim_complaint_category"
)

print("dim_complaint_category created")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_complaint_category created


In [0]:
# ============================================================
# 5. DIM_GEOGRAPHY_BOROUGH_ZIP
# ============================================================

geo_df = (
    silver
    .select(
        F.col("borough_code_std").alias("borough_code"),
        F.col("borough_std").alias("borough"),
        F.col("zip_code_std").alias("zip_code")
    )
    .dropDuplicates()
)

geo_window = Window.orderBy(
    "borough_code",
    "borough",
    "zip_code"
)

geo_df = (
    geo_df
    .withColumn(
        "geography_key",
        F.row_number().over(geo_window)
    )
)

geo_df.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.dim_geography_borough_zip"
)

print("dim_geography_borough_zip created")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_geography_borough_zip created


In [0]:
# ============================================================
# 6. DIM_REQUEST_STATUS
# ============================================================

status_df = (
    silver
    .select(
        F.col("status_std").alias("status"),
        F.col("open_closed_flag").alias("open_closed_flag")
    )
    .dropDuplicates()
)

status_window = Window.orderBy("status", "open_closed_flag")

status_df = (
    status_df
    .withColumn(
        "request_status_key",
        F.row_number().over(status_window)
    )
)

status_df.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.dim_request_status"
)

print("dim_request_status created")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_request_status created


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# ============================================================
# 7. DIM_CHANNEL
# ============================================================

channel_df = (
    silver
    .select(
        F.col("channel_std").alias("channel")
    )
    .dropDuplicates()
)

channel_window = Window.orderBy("channel")

channel_df = (
    channel_df
    .withColumn(
        "channel_key",
        F.row_number().over(channel_window)
    )
)

channel_df.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.dim_channel"
)

print("dim_channel created")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_channel created


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# ============================================================
# 8. DIM_LOCATION_TYPE
# ============================================================

location_df = (
    silver
    .select(
        F.col("location_type_std").alias("location_type")
    )
    .dropDuplicates()
)

location_window = Window.orderBy("location_type")

location_df = (
    location_df
    .withColumn(
        "location_type_key",
        F.row_number().over(location_window)
    )
)

location_df.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.dim_location_type"
)

print("dim_location_type created")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_location_type created


In [0]:
# ============================================================
# DIMENSION CHECK
# ============================================================

dimension_tables = [
    "dim_date",
    "dim_time_band",
    "dim_agency",
    "dim_complaint_category",
    "dim_geography_borough_zip",
    "dim_request_status",
    "dim_channel",
    "dim_location_type"
]

for table in dimension_tables:
    full_name = f"{GOLD_SCHEMA}.{table}"
    count = spark.table(full_name).count()
    print(f"{table}: {count} rows")

dim_date: 365 rows
dim_time_band: 4 rows
dim_agency: 12 rows
dim_complaint_category: 32 rows
dim_geography_borough_zip: 40 rows
dim_request_status: 6 rows
dim_channel: 6 rows
dim_location_type: 8 rows


In [0]:
# ============================================================
# FACT 1 - FACT_SERVICE_REQUEST
# Grain: one trusted request
# ============================================================

s = silver.alias("s")
a = spark.table(f"{GOLD_SCHEMA}.dim_agency").alias("a")
c = spark.table(f"{GOLD_SCHEMA}.dim_complaint_category").alias("c")
g = spark.table(f"{GOLD_SCHEMA}.dim_geography_borough_zip").alias("g")
st = spark.table(f"{GOLD_SCHEMA}.dim_request_status").alias("st")
ch = spark.table(f"{GOLD_SCHEMA}.dim_channel").alias("ch")
lt = spark.table(f"{GOLD_SCHEMA}.dim_location_type").alias("lt")

fact_service_request = (
    s
    .join(
        a,
        F.col("s.agency_code_std") == F.col("a.agency_code"),
        "left"
    )
    .join(
        c,
        (
            (F.col("s.complaint_category_std") == F.col("c.complaint_category")) &
            (F.col("s.complaint_type_std") == F.col("c.complaint_type")) &
            (F.col("s.descriptor_std") == F.col("c.descriptor"))
        ),
        "left"
    )
    .join(
        g,
        (
            (F.col("s.borough_code_std") == F.col("g.borough_code")) &
            (F.col("s.borough_std") == F.col("g.borough")) &
            (F.col("s.zip_code_std") == F.col("g.zip_code"))
        ),
        "left"
    )
    .join(
        st,
        (
            (F.col("s.status_std") == F.col("st.status")) &
            (F.col("s.open_closed_flag") == F.col("st.open_closed_flag"))
        ),
        "left"
    )
    .join(
        ch,
        F.col("s.channel_std") == F.col("ch.channel"),
        "left"
    )
    .join(
        lt,
        F.col("s.location_type_std") == F.col("lt.location_type"),
        "left"
    )
    .select(
        F.col("s.unique_key"),
        F.col("s.physical_record_key"),

        F.col("a.agency_key"),
        F.col("c.complaint_category_key"),
        F.col("g.geography_key"),
        F.col("st.request_status_key"),
        F.col("ch.channel_key"),
        F.col("lt.location_type_key"),

        F.col("s.created_date_key"),
        F.col("s.created_ts"),
        F.col("s.due_ts"),
        F.col("s.closed_ts"),

        F.col("s.priority"),
        F.col("s.resolution_hours"),
        F.col("s.sla_hours_num"),
        F.col("s.sla_eligible_flag"),
        F.col("s.sla_met_flag"),

        F.col("s.request_age_hours"),
        F.col("s.backlog_age_band"),
        F.col("s.route"),
        F.col("s.dq_status")
    )
)

fact_service_request.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.fact_service_request"
)

print("fact_service_request created")
print("Rows:", fact_service_request.count())

fact_service_request created
Rows: 179701


In [0]:
# ============================================================
# FACT 2 - FACT_REQUEST_RESOLUTION
# Grain: one valid resolved request
# ============================================================

fact_request_resolution = (
    silver
    .filter(
        (F.col("closed_ts").isNotNull()) &
        (F.col("created_ts").isNotNull()) &
        (F.col("resolution_hours").isNotNull()) &
        (F.col("resolution_hours") >= 0)
    )
    .select(
        "unique_key",
        "physical_record_key",
        "created_ts",
        "closed_ts",
        "resolution_hours",
        "sla_hours_num",
        "sla_eligible_flag",
        "sla_met_flag"
    )
)

fact_request_resolution.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.fact_request_resolution"
)

print("fact_request_resolution created")
print("Rows:", fact_request_resolution.count())

fact_request_resolution created
Rows: 165200


In [0]:
# ============================================================
# FACT 3 - FACT_REQUEST_BACKLOG_SNAPSHOT
# Grain: one open request per snapshot date
# ============================================================

bounds = (
    silver
    .filter(F.col("created_ts").isNotNull())
    .select(
        F.min(F.to_date("created_ts")).alias("min_date"),
        F.max(
            F.coalesce(
                F.to_date("closed_ts"),
                F.to_date("created_ts")
            )
        ).alias("max_date")
    )
    .collect()[0]
)

min_date = bounds["min_date"]
max_date = bounds["max_date"]

print("Snapshot range:", min_date, "to", max_date)

snapshot_dates = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(min_date).cast("date"),
                F.lit(max_date).cast("date"),
                F.expr("interval 1 day")
            )
        ).alias("snapshot_date")
    )
)

requests_for_snapshot = silver.select(
    "unique_key",
    "physical_record_key",
    "created_ts",
    "closed_ts",
    "request_age_hours",
    "backlog_age_band"
)

fact_request_backlog_snapshot = (
    requests_for_snapshot
    .crossJoin(snapshot_dates)
    .filter(
        (F.to_date("created_ts") <= F.col("snapshot_date")) &
        (
            F.col("closed_ts").isNull() |
            (F.to_date("closed_ts") > F.col("snapshot_date"))
        )
    )
    .select(
        "snapshot_date",
        "unique_key",
        "physical_record_key",
        "created_ts",
        "closed_ts",
        "request_age_hours",
        "backlog_age_band"
    )
    .dropDuplicates(["snapshot_date", "unique_key"])
)

fact_request_backlog_snapshot.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.fact_request_backlog_snapshot"
)

print("fact_request_backlog_snapshot created")
print("Rows:", fact_request_backlog_snapshot.count())

Snapshot range: 2025-01-01 to 2026-01-30
fact_request_backlog_snapshot created
Rows: 2314218


In [0]:
# ============================================================
# FACT 4 - FACT_REQUEST_EVENT_STREAM
# Schema-ready for Week 10 streaming
# ============================================================

event_stream_schema = """
event_id STRING,
event_ts TIMESTAMP,
unique_key STRING,
event_type STRING,
agency_key BIGINT,
complaint_category_key BIGINT,
geography_key BIGINT,
request_status_key BIGINT,
channel_key BIGINT,
location_type_key BIGINT,
payload STRING
"""

empty_event_stream = spark.createDataFrame(
    [],
    event_stream_schema
)

empty_event_stream.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.fact_request_event_stream"
)

print("fact_request_event_stream created")

fact_request_event_stream created


In [0]:
# ============================================================
# SUMMARY 1 - AGENCY SLA SUMMARY
# ============================================================

agency_sla_summary = (
    spark.table(f"{GOLD_SCHEMA}.fact_service_request")
    .groupBy("agency_key")
    .agg(
        F.countDistinct("unique_key").alias("total_requests"),
        F.sum(
            F.when(
                F.col("sla_eligible_flag") == True, 1
            ).otherwise(0)
        ).alias("sla_eligible_requests"),
        F.sum(
            F.when(
                F.col("sla_met_flag") == True, 1
            ).otherwise(0)
        ).alias("sla_met_requests"),
        F.avg(
            F.when(
                F.col("resolution_hours").isNotNull(),
                F.col("resolution_hours")
            )
        ).alias("avg_resolution_hours")
    )
    .withColumn(
        "sla_compliance_rate",
        F.when(
            F.col("sla_eligible_requests") > 0,
            F.col("sla_met_requests") /
            F.col("sla_eligible_requests")
        ).otherwise(F.lit(0.0))
    )
)

agency_sla_summary.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.agency_sla_summary"
)

In [0]:
# ============================================================
# SUMMARY 2 - CATEGORY VOLUME SUMMARY
# ============================================================

category_volume_summary = (
    spark.table(f"{GOLD_SCHEMA}.fact_service_request")
    .groupBy("complaint_category_key")
    .agg(
        F.countDistinct("unique_key").alias("total_requests"),
        F.sum(
            F.when(
                F.col("request_status_key").isNotNull(), 1
            ).otherwise(0)
        ).alias("mapped_requests")
    )
)

category_volume_summary.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.category_volume_summary"
)

In [0]:
# ============================================================
# SUMMARY 3 - BOROUGH BACKLOG SUMMARY
# ============================================================

borough_backlog_summary = (
    spark.table(f"{GOLD_SCHEMA}.fact_request_backlog_snapshot")
    .join(
        spark.table(f"{GOLD_SCHEMA}.fact_service_request")
        .select("unique_key", "geography_key"),
        "unique_key",
        "left"
    )
    .groupBy("snapshot_date", "geography_key")
    .agg(
        F.countDistinct("unique_key").alias("open_backlog")
    )
)

borough_backlog_summary.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.borough_backlog_summary"
)

In [0]:
# ============================================================
# SUMMARY 4 - REQUEST RESOLUTION TREND SUMMARY
# ============================================================

resolution_trend_summary = (
    spark.table(f"{GOLD_SCHEMA}.fact_request_resolution")
    .withColumn(
        "resolution_date",
        F.to_date("closed_ts")
    )
    .groupBy("resolution_date")
    .agg(
        F.countDistinct("unique_key").alias("resolved_requests"),
        F.avg("resolution_hours").alias("avg_resolution_hours"),
        F.min("resolution_hours").alias("min_resolution_hours"),
        F.max("resolution_hours").alias("max_resolution_hours")
    )
)

resolution_trend_summary.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.request_resolution_trend_summary"
)

In [0]:
# ============================================================
# SUMMARY 5 - CHANNEL AND LOCATION SUMMARY
# ============================================================

channel_location_summary = (
    spark.table(f"{GOLD_SCHEMA}.fact_service_request")
    .groupBy(
        "channel_key",
        "location_type_key"
    )
    .agg(
        F.countDistinct("unique_key").alias("total_requests"),
        F.avg("resolution_hours").alias("avg_resolution_hours"),
        F.sum(
            F.when(
                F.col("sla_met_flag") == True, 1
            ).otherwise(0)
        ).alias("sla_met_requests")
    )
)

channel_location_summary.write.mode("overwrite").saveAsTable(
    f"{GOLD_SCHEMA}.channel_and_location_summary"
)

In [0]:
# ============================================================
# GOLD TABLE INVENTORY
# ============================================================

spark.sql(f"SHOW TABLES IN {GOLD_SCHEMA}").show(
    truncate=False
)

+--------+--------------------------------+-----------+
|database|tableName                       |isTemporary|
+--------+--------------------------------+-----------+
|gold    |agency_sla_summary              |false      |
|gold    |borough_backlog_summary         |false      |
|gold    |category_volume_summary         |false      |
|gold    |channel_and_location_summary    |false      |
|gold    |dim_agency                      |false      |
|gold    |dim_channel                     |false      |
|gold    |dim_complaint_category          |false      |
|gold    |dim_date                        |false      |
|gold    |dim_geography_borough_zip       |false      |
|gold    |dim_location_type               |false      |
|gold    |dim_request_status              |false      |
|gold    |dim_time_band                   |false      |
|gold    |fact_request_backlog_snapshot   |false      |
|gold    |fact_request_event_stream       |false      |
|gold    |fact_request_resolution         |false

In [0]:
# ============================================================
# GOLD VALIDATION
# ============================================================

print("========== GOLD VALIDATION ==========")

trusted_count = (
    silver
    .select("unique_key")
    .distinct()
    .count()
)

gold_request_count = (
    spark.table(f"{GOLD_SCHEMA}.fact_service_request")
    .select("unique_key")
    .distinct()
    .count()
)

resolution_count = (
    spark.table(f"{GOLD_SCHEMA}.fact_request_resolution")
    .select("unique_key")
    .distinct()
    .count()
)

snapshot_duplicate_count = (
    spark.table(f"{GOLD_SCHEMA}.fact_request_backlog_snapshot")
    .groupBy("snapshot_date", "unique_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Trusted Silver distinct requests :", trusted_count)
print("Gold fact distinct requests      :", gold_request_count)
print("Resolution fact requests         :", resolution_count)
print("Snapshot duplicate groups        :", snapshot_duplicate_count)

print()
print("Request reconciliation:",
      "PASS" if trusted_count == gold_request_count else "FAIL")

print("Snapshot uniqueness:",
      "PASS" if snapshot_duplicate_count == 0 else "FAIL")

========== GOLD VALIDATION ==========
Trusted Silver distinct requests : 179701
Gold fact distinct requests      : 179701
Resolution fact requests         : 165200
Snapshot duplicate groups        : 0

Request reconciliation: PASS
Snapshot uniqueness: PASS


In [0]:
spark.sql("SHOW TABLES IN cityfix.gold").show(truncate=False)

+--------+--------------------------------+-----------+
|database|tableName                       |isTemporary|
+--------+--------------------------------+-----------+
|gold    |agency_sla_summary              |false      |
|gold    |borough_backlog_summary         |false      |
|gold    |category_volume_summary         |false      |
|gold    |channel_and_location_summary    |false      |
|gold    |dim_agency                      |false      |
|gold    |dim_channel                     |false      |
|gold    |dim_complaint_category          |false      |
|gold    |dim_date                        |false      |
|gold    |dim_geography_borough_zip       |false      |
|gold    |dim_location_type               |false      |
|gold    |dim_request_status              |false      |
|gold    |dim_time_band                   |false      |
|gold    |fact_request_backlog_snapshot   |false      |
|gold    |fact_request_event_stream       |false      |
|gold    |fact_request_resolution         |false

In [0]:
gold_tables = [
    "dim_date",
    "dim_time_band",
    "dim_agency",
    "dim_complaint_category",
    "dim_geography_borough_zip",
    "dim_request_status",
    "dim_channel",
    "dim_location_type",
    "fact_service_request",
    "fact_request_resolution",
    "fact_request_backlog_snapshot",
    "fact_request_event_stream",
    "agency_sla_summary",
    "category_volume_summary",
    "borough_backlog_summary",
    "request_resolution_trend_summary",
    "channel_and_location_summary"
]

for table in gold_tables:
    count = spark.table(f"cityfix.gold.{table}").count()
    print(f"{table}: {count}")

dim_date: 365
dim_time_band: 4
dim_agency: 12
dim_complaint_category: 32
dim_geography_borough_zip: 40
dim_request_status: 6
dim_channel: 6
dim_location_type: 8
fact_service_request: 179701
fact_request_resolution: 165200
fact_request_backlog_snapshot: 2314218
fact_request_event_stream: 0
agency_sla_summary: 12
category_volume_summary: 32
borough_backlog_summary: 15800
request_resolution_trend_summary: 392
channel_and_location_summary: 41


In [0]:
print("========== FINAL GOLD VALIDATION ==========")

# 1. Dimension key uniqueness
dimension_checks = {
    "dim_date": "date_key",
    "dim_time_band": "time_band_key",
    "dim_agency": "agency_key",
    "dim_complaint_category": "complaint_category_key",
    "dim_geography_borough_zip": "geography_key",
    "dim_request_status": "status_key",
    "dim_channel": "channel_key",
    "dim_location_type": "location_type_key"
}

for table, key in dimension_checks.items():
    total = spark.table(f"{GOLD_SCHEMA}.{table}").count()
    distinct = spark.table(f"{GOLD_SCHEMA}.{table}").select(key).distinct().count()
    print(f"{table}: total={total}, distinct_keys={distinct}, PASS={total == distinct}")

========== FINAL GOLD VALIDATION ==========
dim_date: total=365, distinct_keys=365, PASS=True
dim_time_band: total=4, distinct_keys=4, PASS=True
dim_agency: total=12, distinct_keys=12, PASS=True
dim_complaint_category: total=32, distinct_keys=32, PASS=True
dim_geography_borough_zip: total=40, distinct_keys=40, PASS=True


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7197090398879667>, line 17
     15 for table, key in dimension_checks.items():
     16     total = spark.table(f"{GOLD_SCHEMA}.{table}").count()
---> 17     distinct = spark.table(f"{GOLD_SCHEMA}.{table}").select(key).distinct().count()
     18     print(f"{table}: total={total}, distinct_keys={distinct}, PASS={total == distinct}")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1930, in DataFrame._to_table(self)
   1928 def _to_table(self) -> Tuple["pa.Table", Op

In [0]:
print("========== FINAL GOLD VALIDATION ==========")

dimension_checks = {
    "dim_date": "date_key",
    "dim_time_band": "time_band_key",
    "dim_agency": "agency_key",
    "dim_complaint_category": "complaint_category_key",
    "dim_geography_borough_zip": "geography_key",
    "dim_request_status": "request_status_key",
    "dim_channel": "channel_key",
    "dim_location_type": "location_type_key"
}

for table, key in dimension_checks.items():
    df = spark.table(f"{GOLD_SCHEMA}.{table}")

    total = df.count()
    distinct = df.select(key).distinct().count()

    print(
        f"{table}: total={total}, "
        f"distinct_keys={distinct}, "
        f"PASS={total == distinct}"
    )

========== FINAL GOLD VALIDATION ==========
dim_date: total=365, distinct_keys=365, PASS=True
dim_time_band: total=4, distinct_keys=4, PASS=True
dim_agency: total=12, distinct_keys=12, PASS=True
dim_complaint_category: total=32, distinct_keys=32, PASS=True
dim_geography_borough_zip: total=40, distinct_keys=40, PASS=True
dim_request_status: total=6, distinct_keys=6, PASS=True
dim_channel: total=6, distinct_keys=6, PASS=True
dim_location_type: total=8, distinct_keys=8, PASS=True


In [0]:
resolution_bad = spark.table(
    f"{GOLD_SCHEMA}.fact_request_resolution"
).filter(
    F.col("closed_ts") < F.col("created_ts")
).count()

print("Invalid resolution chronology rows:", resolution_bad)
print(
    "Resolution chronology:",
    "PASS" if resolution_bad == 0 else "FAIL"
)

Invalid resolution chronology rows: 0
Resolution chronology: PASS


In [0]:
print("========== SUMMARY RECONCILIATION ==========")

# Agency summary
agency_summary_total = spark.table(
    f"{GOLD_SCHEMA}.agency_sla_summary"
).agg(
    F.sum("total_requests").alias("total")
).collect()[0]["total"]

fact_total = spark.table(
    f"{GOLD_SCHEMA}.fact_service_request"
).select("unique_key").distinct().count()

print(
    "Agency SLA summary total:",
    agency_summary_total,
    "| Fact total:",
    fact_total,
    "| PASS:",
    agency_summary_total == fact_total
)

# Category summary
category_summary_total = spark.table(
    f"{GOLD_SCHEMA}.category_volume_summary"
).agg(
    F.sum("total_requests").alias("total")
).collect()[0]["total"]

print(
    "Category volume summary total:",
    category_summary_total,
    "| Fact total:",
    fact_total,
    "| PASS:",
    category_summary_total == fact_total
)

# Channel + location summary
channel_location_total = spark.table(
    f"{GOLD_SCHEMA}.channel_and_location_summary"
).agg(
    F.sum("total_requests").alias("total")
).collect()[0]["total"]

print(
    "Channel/location summary total:",
    channel_location_total,
    "| Fact total:",
    fact_total,
    "| PASS:",
    channel_location_total == fact_total
)

========== SUMMARY RECONCILIATION ==========
Agency SLA summary total: 179701 | Fact total: 179701 | PASS: True
Category volume summary total: 179701 | Fact total: 179701 | PASS: True
Channel/location summary total: 179701 | Fact total: 179701 | PASS: True


In [0]:
gold_tables = [
    "dim_date",
    "dim_time_band",
    "dim_agency",
    "dim_complaint_category",
    "dim_geography_borough_zip",
    "dim_request_status",
    "dim_channel",
    "dim_location_type",
    "fact_service_request",
    "fact_request_resolution",
    "fact_request_backlog_snapshot",
    "fact_request_event_stream",
    "agency_sla_summary",
    "category_volume_summary",
    "borough_backlog_summary",
    "request_resolution_trend_summary",
    "channel_and_location_summary"
]

display(
    spark.createDataFrame(
        [(t, spark.table(f"cityfix.gold.{t}").count()) for t in gold_tables],
        ["gold_table", "row_count"]
    )
)

gold_table,row_count
dim_date,365
dim_time_band,4
dim_agency,12
dim_complaint_category,32
dim_geography_borough_zip,40
dim_request_status,6
dim_channel,6
dim_location_type,8
fact_service_request,179701
fact_request_resolution,165200
